<a href="https://colab.research.google.com/github/nnott3/KilterTransformer/blob/main/gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/nnott3/KilterTransformer.git
%cd KilterTransformer
!ls

Cloning into 'KilterTransformer'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 237 (delta 32), reused 35 (delta 15), pack-reused 173 (from 1)
Receiving objects: 100% (237/237), 118.34 MiB | 7.96 MiB/s, done.
Resolving deltas: 100% (105/105), done.
Updating files: 100% (56/56), done.
/content/KilterTransformer
bert_improved.ipynb  figs	 models		    req		  uv.lock
bert.ipynb	     gitignore	 project_structure  saved_models
data		     gpt.ipynb	 pyproject.toml     src
EDA.ipynb	     main.ipynb  readme.md	    utils_old


# gpt.py module

In [21]:
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    GPT2LMHeadModel,
    GPT2Config,
    PreTrainedTokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from typing import List, Tuple
import re


class KilterGPT(nn.Module):
    """GPT-2 model for generating Kilter Board climbing routes."""

    def __init__(
        self,
        vocab_size: int,
        n_embd: int = 192,
        n_head: int = 3,
        n_layer: int = 3,
        n_positions: int = 128,
        dropout: float = 0.1
    ):
        super().__init__()
        config = GPT2Config(
            vocab_size=vocab_size,
            n_embd=n_embd,
            n_head=n_head,
            n_layer=n_layer,
            n_positions=n_positions,
            n_ctx=n_positions,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
        )
        self.model = GPT2LMHeadModel(config)
        self.config = config

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=None)

        if labels is not None:
            # Multi-label loss: any remaining token is valid
            logits = outputs.logits
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            loss = 0
            count = 0
            for i in range(shift_labels.size(0)):
                for j in range(shift_labels.size(1)):
                    # Get all valid next tokens (remaining holds)
                    valid_tokens = shift_labels[i, j:]
                    valid_tokens = valid_tokens[valid_tokens != -100]

                    if len(valid_tokens) > 0:
                        log_probs = F.log_softmax(shift_logits[i, j], dim=-1)
                        valid_log_probs = log_probs[valid_tokens]
                        loss -= torch.logsumexp(valid_log_probs, dim=0)
                        count += 1

            outputs.loss = loss / count if count > 0 else loss

        return outputs

    def generate_route(
        self,
        tokenizer: PreTrainedTokenizerFast,
        angle: int = 40,
        grade: int = 18,
        max_length: int = 60,
        temperature: float = 1.0,
        top_k: int = 50,
        top_p: float = 0.95,
        num_return_sequences: int = 1,
        device: str = "cpu",
        logits_processor = None,
        ) -> List[str]:
        self.model.eval()
        self.model.to(device)

        angle_rounded = max(20, min(60, round(angle / 5) * 5))
        grade = max(13, min(27, grade))

        prompt = f"angle{angle_rounded} grade{grade} "
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = self.model.generate(
                input_ids=input_ids,
                max_length=max_length,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                num_return_sequences=num_return_sequences,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                logits_processor=logits_processor,
                repetition_penalty=1.2,
            )

        return [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]

    def validate_route(self, route_str: str) -> Tuple[bool, str]:
        pattern = r"angle(\d+)\s+grade(\d+)\s+(.*)"
        match = re.match(pattern, route_str)

        if not match:
            return False, "Invalid route format"

        angle, grade, holds_str = match.groups()
        angle, grade = int(angle), int(grade)

        if not (20 <= angle <= 60):
            return False, f"Invalid angle: {angle}"
        if not (13 <= grade <= 27):
            return False, f"Invalid grade: {grade}"

        holds = holds_str.split()
        num_start = sum(1 for h in holds if h.startswith("start"))
        num_finish = sum(1 for h in holds if h.startswith("finish"))
        num_hand = sum(1 for h in holds if h.startswith("hand"))
        num_feet = sum(1 for h in holds if h.startswith("feet"))

        if not (1 <= num_start <= 2):
            return False, f"Must have 1-2 start holds, got {num_start}"
        if not (1 <= num_finish <= 2):
            return False, f"Must have 1-2 finish holds, got {num_finish}"
        if len(holds) >= 20:
            return False, f"Too many holds: {len(holds)}"
        if num_hand == 0 and num_feet == 0:
            return False, "Must have at least one hand or feet hold"

        return True, ""


def load_model(model_path: str, device: str = "cpu") -> KilterGPT:
    gpt2_model = GPT2LMHeadModel.from_pretrained(model_path)
    model = KilterGPT(vocab_size=gpt2_model.config.vocab_size)
    model.model = gpt2_model
    model.config = gpt2_model.config
    model.to(device)
    model.eval()
    return model

# tokenizer module

In [22]:
import pprint
from itertools import groupby
import os

# from src.data_processing import DataPreprocessing
from tokenizers import Regex, Tokenizer, models, pre_tokenizers
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import WordLevelTrainer
from transformers import PreTrainedTokenizerFast


def build_vocab():
    # from src.data_processing import HOLD_ID

    special_tokens = ["[PAD]", "[BOS]", "[EOS]", "[UNK]"]
    vocab = {token: idx for idx, token in enumerate(special_tokens)}
    idx = len(vocab)


    for angle in range(20, 61, 5):
        vocab[f"angle{angle}"] = idx
        idx += 1


    for grade in range(13, 28):
        vocab[f"grade{grade}"] = idx
        idx += 1

    # every combos. of hold_id and func
    func = ["feet", "start", "hand", "finish"]
    for hold_id in HOLD_ID:
        for f in func:
            vocab[f"{f}{hold_id}"] = idx
            idx += 1



    print(f"Built vocabulary with {len(vocab)} tokens ({len(vocab) - len(special_tokens)} holds)")
    return vocab


def train_tokenizer(datasets, output_dir, max_length=25):
    """Train and return a tokenizer for the given datasets."""
    special_tokens = {
        "bos_token": "[BOS]",
        "eos_token": "[EOS]",
        "unk_token": "[UNK]",
        "pad_token": "[PAD]",
        }

    # Build pre-defined vocabulary
    vocab = build_vocab()

    # Initialize tokenizer with vocab
    tokenizer = Tokenizer(models.WordLevel(vocab=vocab, unk_token=special_tokens["unk_token"]))
    tokenizer.enable_padding(length=max_length, pad_token=special_tokens["pad_token"])
    tokenizer.enable_truncation(max_length=max_length)

    # Split on underscores and whitespace
    tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Split(Regex(r"_"), behavior="removed"),
        pre_tokenizers.Whitespace()
    ])

    bos_token_id = tokenizer.token_to_id(special_tokens["bos_token"])
    eos_token_id = tokenizer.token_to_id(special_tokens["eos_token"])

    tokenizer.post_processor = TemplateProcessing(
        single=special_tokens["bos_token"] + " $A " + special_tokens["eos_token"],
        special_tokens=[
            (special_tokens["bos_token"], bos_token_id),
            (special_tokens["eos_token"], eos_token_id),
        ],
    )

    inspect_tokenizer(tokenizer)

    # Convert to PreTrainedTokenizerFast
    tokenizer_pretrained = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer,
        model_max_length=max_length,
        padding_side="right",
        truncation_side="right",
        **special_tokens
    )

    # Save
    os.makedirs(output_dir, exist_ok=True)
    print(f"Saving tokenizer to {output_dir}")
    tokenizer_pretrained.save_pretrained(output_dir)

    return tokenizer_pretrained


def inspect_tokenizer(tokenizer):
    vocab_list = list(tokenizer.get_vocab().items())
    print(f"\nVocab size: {len(vocab_list)} tokens")
    print("First 10 tokens:", vocab_list[:10])

    samples = [
        "angle35_grade14_feet1595_start1400",
        "angle40_grade15_feet1595_start1596_hand1597_finish1598",
    ]





# data processing module

In [23]:
"""
Clean data processing for Kilter Board climbs.
Complete implementation with frame parsing, feature engineering, and dataset creation.
"""

import pandas as pd
import numpy as np
import re
import os
import ast
from datasets import Dataset

# Constants
HOLD_ID = list(range(1073, 1396)) + list(range(1447, 1600))

HOLDCOORDINATES = [[1020, 1225], [960, 1225], [900, 1225], [840, 1225], [780, 1225], [720, 1225], [660, 1225], [600, 1225], [540, 1225], [480, 1225], [420, 1225], [360, 1225], [300, 1225], [240, 1225], [180, 1225], [120, 1225], [60, 1225], [60, 1165], [120, 1165], [180, 1165], [240, 1165], [300, 1165], [360, 1165], [420, 1165], [480, 1165], [540, 1165], [600, 1165], [660, 1165], [720, 1165], [780, 1165], [840, 1165], [900, 1165], [960, 1165], [1020, 1165], [60, 1105], [120, 1105], [180, 1105], [240, 1105], [300, 1105], [360, 1105], [420, 1105], [480, 1105], [540, 1105], [600, 1105], [660, 1105], [720, 1105], [780, 1105], [840, 1105], [900, 1105], [960, 1105], [1020, 1105], [60, 1045], [120, 1045], [180, 1045], [240, 1045], [300, 1045], [360, 1045], [420, 1045], [480, 1045], [540, 1045], [600, 1045], [660, 1045], [720, 1045], [780, 1045], [840, 1045], [900, 1045], [960, 1045], [1020, 1045], [60, 985], [120, 985], [180, 985], [240, 985], [300, 985], [360, 985], [420, 985], [480, 985], [540, 985], [600, 985], [660, 985], [720, 985], [780, 985], [840, 985], [900, 985], [960, 985], [1020, 985], [60, 925], [120, 925], [180, 925], [240, 925], [300, 925], [360, 925], [420, 925], [480, 925], [540, 925], [600, 925], [660, 925], [720, 925], [780, 925], [840, 925], [900, 925], [960, 925], [1020, 925], [60, 865], [120, 865], [180, 865], [240, 865], [300, 865], [360, 865], [420, 865], [480, 865], [540, 865], [600, 865], [660, 865], [720, 865], [780, 865], [840, 865], [900, 865], [960, 865], [1020, 865], [60, 805], [120, 805], [180, 805], [240, 805], [300, 805], [360, 805], [420, 805], [480, 805], [540, 805], [600, 805], [660, 805], [720, 805], [780, 805], [840, 805], [900, 805], [960, 805], [1020, 805], [60, 745], [120, 745], [180, 745], [240, 745], [300, 745], [360, 745], [420, 745], [480, 745], [540, 745], [600, 745], [660, 745], [720, 745], [780, 745], [840, 745], [900, 745], [960, 745], [1020, 745], [60, 685], [120, 685], [180, 685], [240, 685], [300, 685], [360, 685], [420, 685], [480, 685], [540, 685], [600, 685], [660, 685], [720, 685], [780, 685], [840, 685], [900, 685], [960, 685], [1020, 685], [60, 625], [120, 625], [180, 625], [240, 625], [300, 625], [360, 625], [420, 625], [480, 625], [540, 625], [600, 625], [660, 625], [720, 625], [780, 625], [840, 625], [900, 625], [960, 625], [1020, 625], [60, 565], [120, 565], [180, 565], [240, 565], [300, 565], [360, 565], [420, 565], [480, 565], [540, 565], [600, 565], [660, 565], [720, 565], [780, 565], [840, 565], [900, 565], [960, 565], [1020, 565], [60, 505], [120, 505], [180, 505], [240, 505], [300, 505], [360, 505], [420, 505], [480, 505], [540, 505], [600, 505], [660, 505], [720, 505], [780, 505], [840, 505], [900, 505], [960, 505], [1020, 505], [60, 445], [120, 445], [180, 445], [240, 445], [300, 445], [360, 445], [420, 445], [480, 445], [540, 445], [600, 445], [660, 445], [720, 445], [780, 445], [840, 445], [900, 445], [960, 445], [1020, 445], [60, 385], [120, 385], [180, 385], [240, 385], [300, 385], [360, 385], [420, 385], [480, 385], [540, 385], [600, 385], [660, 385], [720, 385], [780, 385], [840, 385], [900, 385], [960, 385], [1020, 385], [60, 325], [120, 325], [180, 325], [240, 325], [300, 325], [360, 325], [420, 325], [480, 325], [540, 325], [600, 325], [660, 325], [720, 325], [780, 325], [840, 325], [900, 325], [960, 325], [1020, 325], [60, 265], [120, 265], [180, 265], [240, 265], [300, 265], [360, 265], [420, 265], [480, 265], [540, 265], [600, 265], [660, 265], [720, 265], [780, 265], [840, 265], [900, 265], [960, 265], [1020, 265], [60, 205], [120, 205], [180, 205], [240, 205], [300, 205], [360, 205], [420, 205], [480, 205], [540, 205], [600, 205], [660, 205], [720, 205], [780, 205], [840, 205], [900, 205], [960, 205], [1020, 205], [60, 145], [120, 145], [180, 145], [240, 145], [300, 145], [360, 145], [420, 145], [480, 145], [540, 145], [600, 145], [660, 145], [720, 145], [780, 145], [840, 145], [900, 145], [960, 145], [1020, 145], [1050, 1255], [990, 1255], [930, 1255], [870, 1255], [810, 1255], [750, 1255], [690, 1255], [630, 1255], [570, 1255], [510, 1255], [450, 1255], [390, 1255], [330, 1255], [270, 1255], [210, 1255], [150, 1255], [90, 1255], [30, 1255], [30, 1135], [150, 1135], [270, 1135], [390, 1135], [510, 1135], [630, 1135], [750, 1135], [870, 1135], [990, 1135], [90, 1075], [210, 1075], [330, 1075], [450, 1075], [570, 1075], [690, 1075], [810, 1075], [930, 1075], [1050, 1075], [30, 1015], [150, 1015], [270, 1015], [390, 1015], [510, 1015], [630, 1015], [750, 1015], [870, 1015], [990, 1015], [90, 955], [210, 955], [330, 955], [450, 955], [570, 955], [690, 955], [810, 955], [930, 955], [1050, 955], [30, 895], [150, 895], [270, 895], [390, 895], [510, 895], [630, 895], [750, 895], [870, 895], [990, 895], [90, 835], [210, 835], [330, 835], [450, 835], [570, 835], [690, 835], [810, 835], [930, 835], [1050, 835], [30, 775], [150, 775], [270, 775], [390, 775], [510, 775], [630, 775], [750, 775], [870, 775], [990, 775], [90, 715], [210, 715], [330, 715], [450, 715], [570, 715], [690, 715], [810, 715], [930, 715], [1050, 715], [30, 655], [150, 655], [270, 655], [390, 655], [510, 655], [630, 655], [750, 655], [870, 655], [990, 655], [90, 595], [210, 595], [330, 595], [450, 595], [570, 595], [690, 595], [810, 595], [930, 595], [1050, 595], [30, 535], [150, 535], [270, 535], [390, 535], [510, 535], [630, 535], [750, 535], [870, 535], [990, 535], [90, 475], [210, 475], [330, 475], [450, 475], [570, 475], [690, 475], [810, 475], [930, 475], [1050, 475], [30, 415], [150, 415], [270, 415], [390, 415], [510, 415], [630, 415], [750, 415], [870, 415], [990, 415], [90, 355], [210, 355], [330, 355], [450, 355], [570, 355], [690, 355], [810, 355], [930, 355], [1050, 355], [30, 295], [150, 295], [270, 295], [390, 295], [510, 295], [630, 295], [750, 295], [870, 295], [990, 295]]
HOLDCOORDINATES = [[a, b-112] for a, b in HOLDCOORDINATES]

HOLD_COLORS = {
    'start': "#00DD00",  # Start - Green
    'hand': "#00FFFF",  # Hand - Cyan
    'finish': "#FF00FF",  # Finish - Magenta
    'feet': "#FFA500"   # Foot - Orange
}

ROLE_MAP = {12: "start", 13: "hand", 14: "finish", 15: "feet"}



class DataPreprocessing:
    def __init__(self, csv_dir="csv_exports"):
        self.csv_dir = csv_dir
        self._grade_dict = None

    @property
    def grade_dict(self):
        if self._grade_dict is None:
            try:
                df = pd.read_csv(f"{self.csv_dir}/difficulty_grades.csv")
                self._grade_dict = dict(zip(df["difficulty"], df["boulder_name"]))
            except:
                self._grade_dict = {1: '1a/V0', 2: '1b/V0', 3: '1c/V0',
                                    4: '2a/V0',5: '2b/V0', 6: '2c/V0',
                                    7: '3a/V0', 8: '3b/V0', 9: '3c/V0',
                                    10: '4a/V0', 11: '4b/V0', 12: '4c/V0',
                                    13: '5a/V1', 14: '5b/V1', 15: '5c/V2',
                                    16: '6a/V3', 17: '6a+/V3',18: '6b/V4', 19: '6b+/V4', 20: '6c/V5', 21: '6c+/V5',
                                    22: '7a/V6', 23: '7a+/V7', 24: '7b/V8', 25: '7b+/V8', 26: '7c/V9', 27: '7c+/V10',
                                    28: '8a/V11', 29: '8a+/V12', 30: '8b/V13', 31: '8b+/V14', 32: '8c/V15', 33: '8c+/V16',
                                    34: '9a/V17', 35: '9a+/V18', 36: '9b/V19', 37: '9b+/V20', 38: '9c/V21', 39: '9c+/V22'
                                    }
        return self._grade_dict

    def difficulty_to_v_grade(self, diff_id):
        if pd.isna(diff_id):
            return None
        return int(self.grade_dict.get(int(diff_id), "/?").split('/')[1].split('V')[1])

    def difficulty_to_letter_grade(self, diff_id):
        if pd.isna(diff_id):
            return None
        return self.grade_dict.get(int(diff_id), "/?").split('/')[0]

    def parse_frames(seld, frames, **kwargs): #angle+grade is optional,
        # Transform frames: p1595r12p1596r15 -> start1595_feet1596
        parts = []

        for hold_id, func in re.findall(r"p(\d+)r(\d+)", frames):
            role = ROLE_MAP.get(int(func), "")
            parts.append(f"{role}{hold_id}")

        frames_str = "_".join(parts)
        return_str = ""
        if 'angle' in kwargs:
            return_str += f"angle{int(kwargs['angle'])}_"
        if 'grade' in kwargs:
            return_str += f"grade{round(kwargs['grade'])}_"
        return_str += frames_str
        return return_str



    def load_climbs(self, cache_path="data/climbs_cleaned.csv"):
        """Load climbs as HuggingFace Dataset."""

        if os.path.exists(cache_path):
            df = pd.read_csv(cache_path)
            print(f"Loaded {len(df)} routes from cache {cache_path}")
            # Parse holds_data and add engineered features
            # df['holds_data'] = df['holds_data'].apply(ast.literal_eval)
            # df = self.add_engineered_features(df)

            # Select columns for dataset
            selected_cols = ['name', 'frames', 'holds_data', 'display_difficulty',
                           'angle_y', 'quality_average', 'ascensionist_count',
                           'v_grade', 'letter_grade', 'num_holds', 'num_hand',
                           'num_footonly', 'hand_foot_ratio', 'avg_reach',
                           'max_reach', 'route_area', 'hold_density',
                           'popularity_score', 'angle_x_holds', 'density_x_angle']
            selected_cols = [c for c in selected_cols if c in df.columns]

            dataset = Dataset.from_pandas(df[selected_cols])
            # return dataset.train_test_split(test_size=0.15, seed=42)
            return dataset

    def tokenize_dataset(self, dataset, tokenizer):
        """Tokenize frames column."""
        return dataset.map(lambda example: tokenizer(example["frames"]), batched=True)

    def preprocess_datasets(self, datasets, tokenizer):
        """Tokenize and remove original columns."""
        for name in ("train", "val", "test"):
            col_names = datasets[name].column_names
            datasets[name] = self.tokenize_dataset(datasets[name], tokenizer).remove_columns(col_names)
        return datasets


# real notebook

In [24]:
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    GPT2LMHeadModel,
    GPT2Config,
    PreTrainedTokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from typing import List, Tuple
import re
# from src.data_processing import DataPreprocessing
# from src.tokenizer import train_tokenizer
# from src.gpt import KilterGPT
import numpy as np
from datasets import disable_progress_bar

disable_progress_bar()


run_name = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_DIR = f"models/climb_gpt/{run_name}"
device = "cuda" if torch.cuda.is_available() else "cpu"

dp = DataPreprocessing()
datasets = dp.load_climbs()

# 80, 10, 10 split
train_test = datasets.train_test_split(test_size=0.2, seed=42)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

datasets = {
    'train': train_test['train'],
    'val': val_test['train'],
    'test': val_test['test']
    }

tokenizer = train_tokenizer(datasets, OUT_DIR)
datasets = dp.preprocess_datasets(datasets, tokenizer)

model = KilterGPT(
            vocab_size=tokenizer.vocab_size,
            n_embd=256,
            n_head=4,
            n_layer=6,
            n_positions=128,
            dropout=0.1
        )

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    save_strategy="best",
    save_total_limit=3,
    overwrite_output_dir=True,
    logging_steps=1000,
    num_train_epochs=1,  ####
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    # report_to="tensorboard",
    report_to="none",
    remove_unused_columns=False,
    greater_is_better=False,
    logging_dir=f"{OUT_DIR}/logs",
    load_best_model_at_end=True,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model.model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

trainer.train()

model.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f"\n✓ Model saved to {OUT_DIR}")

test_results = trainer.evaluate(datasets["test"])
print(f"\n✓ Test Loss: {test_results['eval_loss']:.4f}")
print(f"✓ Test Perplexity: {np.exp(test_results['eval_loss']):.2f}")


Loaded 76992 routes from cache data/climbs_cleaned.csv
Built vocabulary with 1932 tokens (1928 holds)

Vocab size: 1932 tokens
First 10 tokens: [('hand1317', 1006), ('feet1168', 408), ('start1579', 1849), ('feet1176', 440), ('feet1532', 1660), ('hand1509', 1570), ('feet1503', 1544), ('feet1498', 1524), ('finish1474', 1431), ('feet1236', 680)]
Saving tokenizer to models/climb_gpt/run_20251031_191357


Map:   0%|          | 0/61593 [00:00<?, ? examples/s]

Map:   0%|          | 0/7699 [00:00<?, ? examples/s]

Map:   0%|          | 0/7700 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
1000,5.852300,5.305619
2000,5.071500,4.833040
3000,4.758500,4.639894



✓ Model saved to models/climb_gpt/run_20251031_191357



✓ Test Loss: 4.5968
✓ Test Perplexity: 99.17
